<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: 'Segoe UI', Tahoma, Arial, sans-serif; font-size: 16px;">

**[🡨 بازگشت به فصل ششم (مدل‌های دیفیوژن و GBM)](Fasl_6_Masterclass.ipynb)**

# 🎓 مسترکلاس مهندسی مالی تصادفی با پایتون
## فصل ۷: مدل‌های پرشی و بازیابی چگالی (Jump Models & Density Recovery)

---
### 🎯 هدف این نوت‌بوک
در فصل قبل دیدیم که حرکت براونی هندسی (GBM) فرض می‌کند قیمت‌ها به صورت پیوسته تغییر می‌کنند. اما در واقعیت، بازارها با اخبار ناگهانی (مثل ورشکستگی‌ها، بحران‌های ژئوپلیتیک یا گزارش‌های اقتصادی) دچار **«پرش» (Jump)** یا سقوط آزاد می‌شوند.
برای رفع این نقص بزرگ، ریاضیدانان مالی فرآیند دیفیوژن (پیوسته) را با فرآیند پواسون (گسسته) ترکیب کردند تا به **مدل‌های پرش-دیفیوژن (Jump-Diffusion Models)** برسند.

در این مسترکلاس آکادمیک موارد زیر را کالبدشکافی می‌کنیم:
1. **مدل مرتون (Merton Model):** مدلی که فرض می‌کند اندازه پرش‌ها دارای توزیع نرمال (لگ‌نرمال برای قیمت) است.
2. **مدل کو (Kou Model):** مدلی که فرض می‌کند اندازه پرش‌ها دارای توزیع نمایی دوگانه نامتقارن (Asymmetric Double Exponential) است (بهتر برای بازارهای دارای چولگی).
3. **بازیابی چگالی با روش کسینوسی (COS Method):** در مدل‌های پرشی، تابع چگالی احتمال (PDF) معمولاً فرمول بسته ندارد. اما ما تابع مشخصه (CF) را داریم. یاد می‌گیریم که چگونه با استفاده از سری‌های فوریه کسینوسی، PDF را بازیابی کنیم.

</div>

In [ ]:
# Install necessary packages for Chapter 7
!pip install scipy numpy pandas matplotlib seaborn scikit-optimize loky

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, poisson, laplace
from scipy.optimize import minimize, shgo
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List

plt.style.use("seaborn-v0_8-darkgrid")
print("\n--- Setup Complete! Libraries for Jump Models are loaded. ---")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۱: زیرساخت شبیه‌سازی مونت‌کارلو (مستقل‌سازی نوت‌بوک)
برای اجرای روان کدها در گوگل کولب، هسته اصلی شبیه‌ساز تصادفی را که در فصول قبل توسعه دادیم، در اینجا بارگذاری می‌کنیم.

</div>

In [ ]:
# --- Base Monte Carlo Infrastructure (from Chapters 4 & 5) ---

class TargetSamplingDensity(ABC):
    @abstractmethod
    def pdf(self, x): ...
    @abstractmethod
    def sample(self, n_vars, n_sample_paths=1): ...

class MonteCarloSimulation:
    @dataclass
    class MCEstimate:
        samples: List = None
        mean: np.ndarray = None
        standard_error: np.ndarray = None

    def __init__(self, h_x_fun, target_sampling_density, n_vars, n_sample_paths):
        self.h_x_fun = h_x_fun
        self.target_sampling_density = target_sampling_density
        self.n_vars = n_vars
        self.n_sample_paths = n_sample_paths

    def new_estimate(self):
        # Sample from the target density
        z = self.target_sampling_density.sample(self.n_vars, self.n_sample_paths)
        # Apply path generation function
        samples = np.vectorize(lambda y: self.h_x_fun(y), otypes=[float])(z)
        mean = np.average(samples, axis=0)
        variance = np.sum(np.power(samples - mean, 2), axis=0) / (self.n_sample_paths - 1 + 1e-10)
        return self.MCEstimate(samples=samples, mean=mean, standard_error=np.sqrt(variance / self.n_sample_paths))

class ForecastingProcess(ABC):
    def __init__(self, n_sample_paths, initial_state, sampling_density):
        self._n_sample_paths = n_sample_paths
        self._initial_state = initial_state
        self._state_t = initial_state
        self._sampling_density = sampling_density
        self._t, self._T = 0, 0

    def forecast(self, T):
        self._T, self._t = T, 0
        mcs = MonteCarloSimulation(self._update_current_state, self._sampling_density, T, self._n_sample_paths)
        e = mcs.new_estimate()
        self._state_t = self._initial_state
        return e

    @abstractmethod
    def _update_current_state(self, z): ...

    def _reset_new_sample_path_state(self):
        if self._t >= self._T:
            self._state_t = self._initial_state
            self._t = 0
        self._t += 1

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۲: ریاضیات مدل پرش-دیفیوژن پایه (Base Jump-Diffusion)

معادله دیفرانسیل تصادفی (SDE) برای یک مدل پرش-دیفیوژن به شکل زیر است:
$$ \frac{dS_t}{S_{t-1}} = \mu dt + \sigma dW_t + J_t dN_t $$

که در آن:
* $\sigma dW_t$: بخش پیوسته بازار (نویز نرمال).
* $dN_t$: فرآیند پواسون با نرخ $\lambda$ که نشان‌دهنده زمان وقوع پرش است (تعداد پرش‌ها در واحد زمان).
* $J_t$: متغیر تصادفی نشان‌دهنده **اندازه پرش (Jump Size)**.

در کد زیر، کلاس `ParametricJumpProcess` این فرآیند را مدیریت می‌کند. در هر گام زمانی ما سه متغیر تصادفی تولید می‌کنیم: نویز پیوسته ($dW$)، وقوع پرش ($dJ_p$ از توزیع پواسون) و اندازه پرش ($dJ$).

</div>

In [ ]:
# --- Base Jump Process Infrastructure ---

class ParametricJumpProcessSamplingDensity(TargetSamplingDensity, ABC):
    """
    Base class to sample the required random variables for a jump process:
    1. dW: Standard Normal for Diffusion
    2. dJ_p: Poisson for jump occurrence
    3. dJ: Jump size (distribution depends on Merton or Kou)
    """
    def __init__(self, r, σ, λ):
        self._r = r
        self._σ = σ
        self._λ = λ

    def pdf(self, x):
        return None

    def sample(self, n_vars, n_sample_paths=1):
        # 1. Continuous Diffusion Noise
        dWs = norm.rvs(size=(n_sample_paths, n_vars))
        # 2. Jump Arrival (Poisson)
        dJ_p = poisson.rvs(mu=self._λ, size=(n_sample_paths, n_vars))
        # 3. Jump Sizes
        dJ = self._sample_jumps(n_sample_paths, n_vars)

        # Package them into a structured array of dictionaries for the simulator
        return np.array([[{'dW': w, 'dJ_p': j_p, 'dJ': j}
                          for w, j, j_p in zip(dw, dj, dj_p)]
                         for dw, dj, dj_p in zip(dWs, dJ, dJ_p)])

    @abstractmethod
    def _sample_jumps(self, n_sample_paths, n_jumps): ...

class ParametricJumpProcess(ForecastingProcess, ABC):
    def __init__(self, r, σ, λ, sampling_density, initial_state=1.0, n_sample_paths=5):
        self._r = r
        self._σ = σ
        self._λ = λ
        self._μ = self._compute_jump_drift() # Compensator to ensure martingale property

        super().__init__(n_sample_paths, initial_state=initial_state, sampling_density=sampling_density)

    @abstractmethod
    def _compute_jump_drift(self): ...

    def _update_current_state(self, z):
        self._reset_new_sample_path_state()
        dW = z['dW']
        dJ = z['dJ']
        dJ_p = z['dJ_p']

        # S_t = S_{t-1} * exp( μ + σ*dW + dJ*dJ_p )
        self._state_t = self._state_t * np.exp(self._μ + (self._σ * dW) + (dJ * dJ_p))
        return self._state_t

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۳: مدل پرشی مرتون (Merton's Jump-Diffusion Model)

در سال ۱۹۷۶، مرتون مدل بلک-شولز را گسترش داد. او فرض کرد که وقتی پرشی رخ می‌دهد، اندازه بازده لگاریتمی آن پرش از یک **توزیع نرمال** پیروی می‌کند:
$$ \ln(1 + J) \sim \mathcal{N}(\mu_j, \sigma_j^2) $$

برای اینکه مدل آربیتراژ (Arbitrage-Free) بماند، ما باید دریفت پیوسته را اصلاح کنیم (Compensator). رابطه ریاضی دریفت در مدل مرتون:
$$ \mu_{merton} = r - \frac{\sigma^2}{2} - \lambda \left( e^{\mu_j + \frac{\sigma_j^2}{2}} - 1 \right) $$

</div>

In [ ]:
# --- Merton Jump-Diffusion Model ---

class MertonProcessSamplingDensity(ParametricJumpProcessSamplingDensity):
    def __init__(self, r, σ, λ, μ_j, σ_j):
        self._μ_j = μ_j
        self._σ_j = σ_j
        super().__init__(r=r, σ=σ, λ=λ)

    def _sample_jumps(self, n_sample_paths, n_jumps):
        # Jump sizes are Normally distributed in Merton's model
        return norm.rvs(size=(n_sample_paths, n_jumps), loc=self._μ_j, scale=self._σ_j)

class MertonProcess(ParametricJumpProcess):
    def __init__(self, r, σ, λ, μ_j, σ_j, initial_state=1.0, n_sample_paths=5):
        self._μ_j = μ_j
        self._σ_j = σ_j
        super().__init__(r=r, σ=σ, λ=λ,
                         sampling_density=MertonProcessSamplingDensity(r, σ, λ, μ_j, σ_j),
                         initial_state=initial_state,
                         n_sample_paths=n_sample_paths)

    def _compute_jump_drift(self):
        # Mathematical compensator for Merton Model
        return self._r - (0.5 * (self._σ**2)) - (self._λ * (np.exp(self._μ_j + 0.5 * (self._σ_j**2)) - 1))

# ----------------- Simulation & Visualization -----------------
def plot_jump_process_paths(forecast_result, title):
    samples = pd.DataFrame(forecast_result.samples).transpose()
    mean_path = pd.DataFrame(forecast_result.mean)

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    samples.plot(ax=ax[0], alpha=0.6, legend=False, color="teal")
    ax[0].set_title(f"{title}: Monte Carlo Sample Paths")
    ax[0].set_ylabel("Asset Price S(t)")

    mean_path.plot(ax=ax[1], color='crimson', linewidth=2, legend=False)
    ax[1].set_title("Expected Mean Path")
    plt.tight_layout()
    plt.show()

print("Simulating Merton Jump-Diffusion Process (Notice the sudden sharp jumps):")
# λ=0.05 implies low frequency of jumps, but σ_j=0.2 implies large impact when they happen
merton = MertonProcess(r=0.01, σ=0.03, λ=0.05, μ_j=-0.1, σ_j=0.2,
                       initial_state=100, n_sample_paths=20)
plot_jump_process_paths(merton.forecast(250), "Merton Model")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۴: مدل پرشی کو (Kou's Jump-Diffusion Model)

استیون کو (S. Kou) مدل مرتون را ارتقا داد. در بازارهای مالی واقعی، سقوط‌های شدید (Crashes) بیشتر و بزرگتر از صعودهای ناگهانی (Rallies) هستند. توزیع نرمال در مدل مرتون تقارن دارد و این عدم تقارن را خوب نشان نمی‌دهد.

مدل کو فرض می‌کند که اندازه پرش‌ها دارای **توزیع نمایی دوگانه نامتقارن (Asymmetric Double Exponential)** است.
* با احتمال $p$ یک پرش مثبت (صعود) با توزیع نمایی دارای پارامتر $\alpha_1$ رخ می‌دهد.
* با احتمال $1-p$ یک پرش منفی (سقوط) با توزیع نمایی دارای پارامتر $\alpha_2$ رخ می‌دهد.

محاسبه دریفت اصلاح‌شده در مدل کو:
$$ \mu_{kou} = r - \frac{\sigma^2}{2} + \lambda \left( 1 - \frac{p \alpha_1}{\alpha_1 - 1} - \frac{(1-p) \alpha_2}{\alpha_2 + 1} \right) $$

</div>

In [ ]:
# --- Kou Jump-Diffusion Model ---

class KouProcessSamplingDensity(ParametricJumpProcessSamplingDensity):
    def __init__(self, r, σ, λ, p, α_1, α_2):
        self._p = p
        self._α_1 = α_1
        self._α_2 = α_2
        super().__init__(r=r, σ=σ, λ=λ)

    def _sample_jumps(self, n_sample_paths, n_jumps):
        # Simplified approach to sample from Asymmetric Double Exponential
        total_samples = n_sample_paths * n_jumps
        u = np.random.uniform(0, 1, total_samples)
        # Inverse transform sampling for Laplace / Double Exponential
        jumps = np.where(u < self._p,
                         -np.log(1 - u / self._p) / self._α_1,
                         np.log((1 - u) / (1 - self._p)) / self._α_2)
        return jumps.reshape(n_sample_paths, n_jumps)

class KouProcess(ParametricJumpProcess):
    def __init__(self, r, σ, λ, p, α_1, α_2, initial_state=1.0, n_sample_paths=5):
        self._p, self._α_1, self._α_2 = p, α_1, α_2
        super().__init__(r=r, σ=σ, λ=λ,
                         sampling_density=KouProcessSamplingDensity(r, σ, λ, p, α_1, α_2),
                         initial_state=initial_state,
                         n_sample_paths=n_sample_paths)

    def _compute_jump_drift(self):
        # Mathematical compensator for Kou Model
        de_term_1 = self._p * self._α_1 / (self._α_1 - 1)
        de_term_2 = (1 - self._p) * self._α_2 / (self._α_2 + 1)
        return self._r - (0.5 * (self._σ**2)) + (self._λ * (1 - de_term_1 - de_term_2))

print("Simulating Kou Jump-Diffusion Process (Notice the asymmetric nature of crashes):")
# p=0.3 implies 70% of jumps are negative. α_2=1.5 means negative jumps are large.
kou = KouProcess(r=0.01, σ=0.03, λ=0.05, p=0.3, α_1=3.0, α_2=1.5,
                 initial_state=100, n_sample_paths=20)
plot_jump_process_paths(kou.forecast(250), "Kou Model (Asymmetric)")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۵: بازیابی چگالی (Density Recovery) با روش COS

در فصل قبل (فصل ۶)، برای تخمین پارامترهای مدل با روش MLE، مستقیماً از تابع چگالی (PDF) توزیع نرمال استفاده کردیم. اما در مدل‌های مرتون و کو، **توزیع تغییرات قیمت یک فرمول جبری و مشخص (Closed-form PDF) ندارد!**

با این حال، **تابع مشخصه (Characteristic Function - $\phi(\omega)$)** آن‌ها در فضای اعداد مختلط مشخص است. پرفسور اوسترلی (Oosterlee) روش **COS Method** را ابداع کرد که به ما اجازه می‌دهد تابع چگالی ناشناخته $f(x)$ را با استفاده از سری فوریه کسینوسی و تابع مشخصه تقریب بزنیم:

$$ f(x) \approx \frac{2}{b-a} \sum_{k=0}^{N-1} \sideset{''}{} \text{Re} \left[ \phi\left(\frac{k\pi}{b-a}\right) \exp\left(-i \frac{k\pi a}{b-a}\right) \right] \cos\left(k\pi \frac{x-a}{b-a}\right) $$

در کد زیر، این شاهکار ریاضیات محاسباتی را پیاده‌سازی کرده و نشان می‌دهیم چگونه از تابع مشخصه (CF) به تابع چگالی (PDF) پل می‌زنیم.

</div>

In [ ]:
# --- Density Recovery via Fourier-Cosine (COS) Method ---

class COSMethodBasedDensityRecovery:
    """
    Recovers the Probability Density Function (PDF) from a Characteristic Function (CF)
    using Fourier-Cosine series expansion.
    """
    def __init__(self, N_freq, ϕ_ω: callable):
        self._N_freq = N_freq
        self._ϕ_ω = ϕ_ω
        self._k = np.arange(N_freq)

    def recover(self, x_i, t, cumulants: dict, θ):
        # 1. Determine integration range [a, b] based on cumulants (mean and variance)
        intg_range = 8.0
        if cumulants['κ2'] > 0:
            intg_range = 8 * np.sqrt(cumulants['κ2'] + np.sqrt(cumulants.get('κ4', 0)))

        a = x_i + cumulants['κ1'] - intg_range
        b = x_i + cumulants['κ1'] + intg_range
        d = b - a

        # 2. Compute Cosine expansion coefficients f_k
        u = (self._k * np.pi) / d
        # Re[ CF(u) * exp(-i * a * u) ]
        f_k = (2.0 / d) * (self._ϕ_ω(u, θ, t) * np.exp(-1j * a * u)).real
        f_k[0] = f_k[0] * 0.5  # First term adjustment in Fourier series

        # 3. Reconstruct Density at x_i
        density = np.sum(f_k * np.cos(self._k * np.pi * (x_i - a) / d))
        return np.abs(density)

# Let's test the COS method on a known distribution (Gaussian) to prove its magic
def _gaussian_cf(ω, θ, t):
    mu, σ = θ
    return np.exp(1j * mu * ω - 0.5 * (σ**2) * (ω**2))

def test_cos_recovery():
    θ_test = (3.0, 1.2) # Mean=3.0, StdDev=1.2
    x_range = np.linspace(0, 6, 100)

    # Recover using COS method
    cos_recover = COSMethodBasedDensityRecovery(N_freq=500, ϕ_ω=_gaussian_cf)
    cumulants = {'κ1': 3.0, 'κ2': 1.2**2, 'κ4': 0.0}

    recovered_density = [cos_recover.recover(x, t=1.0, cumulants=cumulants, θ=θ_test) for x in x_range]
    true_density = norm.pdf(x_range, loc=3.0, scale=1.2)

    plt.figure(figsize=(10, 5))
    plt.plot(x_range, true_density, label='True Gaussian PDF', color='black', linewidth=3, alpha=0.5)
    plt.plot(x_range, recovered_density, label='Recovered via COS Method', color='crimson', linestyle='--')
    plt.title("Density Recovery Proof: COS Method vs True PDF")
    plt.legend()
    plt.show()

print("اثبات عملکرد روش کسینوسی در بازیابی چگالی:")
test_cos_recovery()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

---
### 🏁 نتیجه‌گیری فصل ۷
در این فصل ما مرزهای شبیه‌سازی مالی را به شدت جابجا کردیم:
1. دریافتیم که بازارهای واقعی پیوسته نیستند و نیاز به ترکیب حرکت براونی با فرآیندهای پرشی پواسون داریم.
2. دو مدل استاندارد صنعتی یعنی **مرتون** (پرش‌های نرمال) و **کو** (پرش‌های نامتقارن نمایی) را با کدهای کاملاً شی‌گرا پیاده‌سازی کردیم.
3. با یک چالش بزرگ مواجه شدیم: عدم وجود فرمول تابع چگالی (PDF) برای مدل‌های پرشی. ما این چالش را با کمک تکنیک ریاضیاتی خیره‌کننده **Oosterlee (روش کسینوسی)** حل کردیم و نشان دادیم چگونه می‌توان چگالی را از روی تابع مشخصه در فضای فرکانس بازیابی نمود.

این مدل‌ها پایه و اساس سیستم‌های نوین قیمت‌گذاری در فصل هشتم (Options Pricing) خواهند بود.

</div>